# Telecom Churn Prediction

**Name:** Sunday Babatunde  
**Fellow ID:** FE/26/2850020886  
**3MTT Email:** 4peace1@gmail.com  
**Gender:** Male  
**Cohort:** NextGen Cohort, Lagos

## Project summary

I built this notebook to see whether customer information could help a telecom retention team decide who to contact first. I kept separate training, validation and test sets so that the final result would be based on customers the model had not seen. Because missing a real churner may be costly, I also tested a lower decision threshold instead of accepting the usual 50% cut-off.

## My approach

### Decisions I made

- I treated `Churn = Yes` as the outcome the model should find.
- I handled missing `TotalCharges` values inside the pipeline, so the same rule is used during training and prediction.
- I placed more importance on finding actual churners, even though this means contacting some customers who would have stayed.
- The dataset is a public learning sample, so I do not present its patterns as evidence about a Nigerian operator.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, classification_report,
                             confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_PATH = Path('Telecom Churn Model1.csv')

## Data

### 1. Load and validate the input

In [ ]:
df = pd.read_csv(DATA_PATH)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print('Shape:', df.shape)
print('Duplicate rows:', df.duplicated().sum())
print('Missing TotalCharges:', df['TotalCharges'].isna().sum())
print('Churn rate:', f"{(df['Churn'] == 'Yes').mean():.2%}")
df.head()

### 2. Explore decision-relevant segments

In [ ]:
eda = df.assign(ChurnFlag=(df['Churn'] == 'Yes').astype(int))
contract_churn = eda.groupby('Contract')['ChurnFlag'].agg(['mean', 'count']).sort_values('mean', ascending=False)
contract_churn.style.format({'mean': '{:.1%}'})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
contract_churn['mean'].sort_values().plot.barh(ax=axes[0], color='#2563eb')
axes[0].set(title='Observed churn rate by contract', xlabel='Churn rate', ylabel='Contract')
axes[0].xaxis.set_major_formatter(lambda x, pos: f'{x:.0%}')
for status, color in [(0, '#64748b'), (1, '#f59e0b')]:
    axes[1].hist(eda.loc[eda.ChurnFlag == status, 'tenure'], bins=18, alpha=.65,
                 label='Churned' if status else 'Stayed', color=color)
axes[1].set(title='Customer tenure by outcome', xlabel='Tenure (months)', ylabel='Customers')
axes[1].legend()
plt.tight_layout(); plt.show()

## Results

### 3. Build a leakage-safe pipeline and data split

In [ ]:
X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn'].map({'Yes': 1, 'No': 0})
numeric = X.select_dtypes(include=np.number).columns.tolist()
categorical = X.columns.difference(numeric).tolist()
preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric),
    ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                              ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical),
])
def make_model():
    return Pipeline([('preprocessor', preprocessor),
                     ('classifier', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))])

X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=.20, stratify=y, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=.25, stratify=y_dev, random_state=RANDOM_STATE)
print('Train / validation / test:', len(X_train), len(X_val), len(X_test))

### 4. Choose a practical review threshold

In [ ]:
threshold_model = make_model().fit(X_train, y_train)
validation_probability = threshold_model.predict_proba(X_val)[:, 1]
rows = []
for threshold in np.arange(.20, .61, .01):
    prediction = validation_probability >= threshold
    rows.append({'threshold': threshold, 'precision': precision_score(y_val, prediction),
                 'recall': recall_score(y_val, prediction), 'f1': f1_score(y_val, prediction)})
threshold_results = pd.DataFrame(rows)
eligible = threshold_results[threshold_results.recall >= .70]
operating_threshold = float(eligible.loc[eligible.f1.idxmax(), 'threshold'])
print('Selected threshold:', f'{operating_threshold:.0%}')
threshold_results.iloc[(threshold_results.threshold - operating_threshold).abs().argsort()[:5]].sort_values('threshold')

### 5. Refit on development data and evaluate once on holdout data

In [ ]:
final_model = make_model().fit(X_dev, y_dev)
test_probability = final_model.predict_proba(X_test)[:, 1]
test_prediction = test_probability >= operating_threshold
metrics = pd.Series({
    'Accuracy': accuracy_score(y_test, test_prediction),
    'Precision': precision_score(y_test, test_prediction),
    'Recall': recall_score(y_test, test_prediction),
    'F1': f1_score(y_test, test_prediction),
    'ROC-AUC': roc_auc_score(y_test, test_probability),
    'Average precision': average_precision_score(y_test, test_probability),
})
metrics.to_frame('Holdout result').style.format('{:.2%}')

In [ ]:
cm = confusion_matrix(y_test, test_prediction)
print(classification_report(y_test, test_prediction, target_names=['Stayed', 'Churned']))
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2): ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=12)
ax.set(xticks=[0,1], yticks=[0,1], xticklabels=['Stayed','Churned'], yticklabels=['Stayed','Churned'],
       xlabel='Predicted', ylabel='Actual', title='Confusion matrix on holdout data')
fig.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

## What I learned

- The model does a reasonable job of ranking higher-risk customers, with a test ROC-AUC of about 84%.
- Moving the review threshold to 33% helped the model find about 72% of the actual churners in the test data.
- That improvement comes with extra false alarms, so a real company would need to choose a threshold that fits its budget and retention-team capacity.
- Contract type, tenure, payment method and support services were useful clues, but they do not prove why a customer left.
- Before using a model like this in practice, I would retrain it on recent local data, check fairness and calibration, protect customer information and monitor how performance changes over time.